In [1]:
import os
import shutil
from pathlib import Path

# Define workspace directory paths
WORKING_DIR = Path("/kaggle/working")
REPO_NAME = "clickbait-spoiling-IR-final-project"
REPO_DIR = WORKING_DIR / REPO_NAME
REPO_URL = f"https://github.com/aryonmt/{REPO_NAME}.git"

def setup_workspace():
    """Clones the repository and configures the current active working directory."""
    # Clean up existing repository directory to ensure a fresh, idempotent run
    if REPO_DIR.exists():
        print(f"Cleaning existing directory: {REPO_DIR}")
        shutil.rmtree(REPO_DIR)
    
    # Clone the repository directly to the Kaggle working workspace
    print(f"Cloning repository from {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    
    # Verify the directory structure is fully resolved
    if REPO_DIR.exists():
        print(f"Workspace resolved successfully. Changing directory to {REPO_DIR}")
        os.chdir(REPO_DIR)
    else:
        raise FileNotFoundError(f"Failed to clone repository into {REPO_DIR}")

setup_workspace()

Cloning repository from https://github.com/aryonmt/clickbait-spoiling-IR-final-project.git...


Cloning into '/kaggle/working/clickbait-spoiling-IR-final-project'...


Workspace resolved successfully. Changing directory to /kaggle/working/clickbait-spoiling-IR-final-project


In [2]:
from pathlib import Path

# Define Kaggle input source paths
PRIMARY_TRAIN_SOURCE = Path("/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/train.jsonl")
PRIMARY_VAL_SOURCE = Path("/kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/validation.jsonl")

FALLBACK_TRAIN_SOURCE = Path("/kaggle/input/click-bait-spoiling-dataset/train.jsonl")
FALLBACK_VAL_SOURCE = Path("/kaggle/input/click-bait-spoiling-dataset/validation.jsonl")

PROJECT_DATA_DIR = Path("./data")

def align_datasets():
    """Links external Kaggle dataset splits to the project's internal data directory."""
    # Ensure local project data folder exists
    PROJECT_DATA_DIR.mkdir(parents=True, exist_ok=True)
    
    train_source = PRIMARY_TRAIN_SOURCE
    val_source = PRIMARY_VAL_SOURCE
    
    # Fallback resolution check in case Kaggle mount directory shifts
    if not train_source.exists() or not val_source.exists():
        print("[WARNING] Primary dataset path not found. Checking fallback path...")
        train_source = FALLBACK_TRAIN_SOURCE
        val_source = FALLBACK_VAL_SOURCE
        
    if not train_source.exists() or not val_source.exists():
        raise FileNotFoundError("Could not resolve dataset paths in Kaggle inputs directory.")

    train_target = PROJECT_DATA_DIR / "train.jsonl"
    val_target = PROJECT_DATA_DIR / "validation.jsonl"
    
    # Clean up existing symbolic links or stale files to maintain idempotency
    if train_target.is_symlink() or train_target.exists():
        train_target.unlink()
    if val_target.is_symlink() or val_target.exists():
        val_target.unlink()
        
    # Generate symbolic links
    train_target.symlink_to(train_source)
    val_target.symlink_to(val_source)
    
    print("Dataset successfully mapped to project workspace:")
    print(f" - Train Link: {train_target.resolve()} -> {train_target.readlink()}")
    print(f" - Val Link: {val_target.resolve()} -> {val_target.readlink()}")

align_datasets()

Dataset successfully mapped to project workspace:
 - Train Link: /kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/train.jsonl -> /kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/train.jsonl
 - Val Link: /kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/validation.jsonl -> /kaggle/input/datasets/amirrezanemati05/click-bait-spoiling-dataset/validation.jsonl


In [3]:
import os
import importlib

# Define key packages to verify post-installation
REQUIRED_PACKAGES = ["transformers", "datasets", "torch", "evaluate", "sentencepiece"]

def install_dependencies():
    """Installs required packages from the project's requirements.txt file."""
    print("Installing requirements from requirements.txt...")
    exit_code = os.system("pip install -r requirements.txt --quiet")
    if exit_code != 0:
        raise RuntimeError("Failed to install required dependencies.")
    print("[SUCCESS] Dependencies installed.")

def verify_installations():
    """Imports and prints installed versions of core project dependencies."""
    print("\nVerifying installed package versions:")
    for package in REQUIRED_PACKAGES:
        try:
            module = importlib.import_module(package)
            version = getattr(module, "__version__", "unknown")
            print(f" - {package}: {version}")
        except ImportError:
            print(f" - [ERROR] {package} is not installed.")

install_dependencies()
verify_installations()

Installing requirements from requirements.txt...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.26 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ydf 

[SUCCESS] Dependencies installed.

Verifying installed package versions:
 - transformers: 5.0.0
 - datasets: 4.8.5
 - torch: 2.10.0+cu128
 - evaluate: 0.4.6
 - sentencepiece: 0.2.1


In [4]:
import torch

def verify_gpu_acceleration():
    """Checks PyTorch CUDA support and displays specifications of the active GPU."""
    cuda_active = torch.cuda.is_available()
    print("=" * 60)
    print(f"CUDA GPU Acceleration Active: {cuda_active}")
    print("=" * 60)
    
    if cuda_active:
        device_name = torch.cuda.get_device_name(0)
        allocated_memory = torch.cuda.memory_allocated(0) / (1024 ** 2)
        print(f"Device Name: {device_name}")
        print(f"Allocated VRAM: {allocated_memory:.2f} MB")
    else:
        print("[WARNING] Running on CPU mode. Transformer training will be extremely slow.")
        print("Go to Notebook settings (right panel) -> Accelerator -> Select 'T4 GPU'.")

verify_gpu_acceleration()

CUDA GPU Acceleration Active: True
Device Name: Tesla T4
Allocated VRAM: 0.00 MB


In [5]:
import os

def run_eda_analysis():
    """Runs the main exploratory data analysis script on the Kaggle train split."""
    print("Launching Exploratory Data Analysis (EDA)...")
    exit_code = os.system("python main_eda.py")
    if exit_code != 0:
        raise RuntimeError("EDA processing failed.")
    
    print("[SUCCESS] EDA analysis complete. Generated artifacts in reports directory:")
    # Verify the generated plots in the output directory
    os.system("ls -la reports/")

run_eda_analysis()

Launching Exploratory Data Analysis (EDA)...
Reading data from data/train.jsonl...

=== Running Dataset Sanity Checks ===
Total Rows: 3200
Total Columns: 14
[SUCCESS] All essential columns are present.

Missing Values Count per Critical Column:
uuid                0
postText            0
targetParagraphs    0
tags                0
dtype: int64

Calculating word counts for sequences...

=== Sequence Length Statistical Summary ===
       post_word_count  article_word_count
count      3200.000000         3200.000000
mean         10.790938          514.762812
std           3.483859          583.351337
min           1.000000            7.000000
50%          11.000000          352.500000
75%          13.000000          642.250000
90%          15.000000         1033.100000
95%          17.000000         1396.100000
99%          20.000000         2372.010000
max          28.000000        14059.000000

[SUCCESS] Distribution plots saved to: reports/sequence_length_distributions.png

=== Class D

In [6]:
import os

def train_task1_classifier():
    """Fine-tunes the Task 1 sequence classifier using the defaults configured in config.py."""
    print("Starting Task 1 Sequence Classifier training...")
    
    # We let main_transformer_task1.py automatically read model_name, lr, 
    # output_dir, and epochs from the updated PipelineConfig in config.py.
    cmd = (
        "python main_transformer_task1.py "
        "--train_path data/train.jsonl "
        "--val_path data/validation.jsonl"
    )
    
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Task 1 Sequence Classifier training pipeline failed.")
        
    print("[SUCCESS] Task 1 sequence classification fine-tuning completed safely.")

train_task1_classifier()

Starting Task 1 Sequence Classifier training...


Reading data from data/train.jsonl...
Reading data from data/validation.jsonl...
Mapping and processing partitions for: roberta-base


Map: 100%|██████████| 800/800 [00:00<00:00, 1178.31 examples/s]


[INFO] Computed Dynamic Class Weights: [np.float64(0.7802974884174592), np.float64(0.837257980115123), np.float64(1.9081693500298151)]


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1624.82it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

...

{'train_runtime': '955.5', 'train_samples_per_second': '16.75', 'train_steps_per_second': '0.523', 'train_loss': '1.489', 'epoch': '5'}

=== RUNNING CONVERGENCE EVALUATION ===


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.6741535663604736, 'eval_f1': 0.7177123042029607, 'eval_runtime': 13.9387, 'eval_samples_per_second': 57.394, 'eval_steps_per_second': 3.587, 'epoch': 5.0}
Saving optimized Task 1 classifier weights to outputs...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s]


Removed intermediate checkpoint folder: outputs/checkpoint-400
[SUCCESS] Task 1 sequence classification fine-tuning completed safely.


In [7]:
import os

def train_phrase_qa_model():
    """Fine-tunes the extractive QA model on phrase-type spoilers using config.py defaults."""
    print("Starting Task 2 Phrase QA model training...")
    cmd = (
        "python main_transformer_task2_train.py "
        "--spoiler_type phrase "
        "--train_path data/train.jsonl "
        "--val_path data/validation.jsonl"
    )
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Phrase QA model fine-tuning failed.")
    print("[SUCCESS] Phrase QA model fine-tuning completed successfully.")

train_phrase_qa_model()

Starting Task 2 Phrase QA model training...


Loading and filtering phrase datasets...
Reading data from data/train.jsonl...
Filtered dataset from 3200 to 1367 samples of type 'phrase'.
Reading data from data/validation.jsonl...
Filtered dataset from 800 to 335 samples of type 'phrase'.
Tokenizing and mapping text spans using: deepset/roberta-base-squad2


Map:   0%|          | 0/1367 [00:00<?, ? examples/s]2026-07-04 23:37:41 - qa_preprocessor - INFO - Data preprocessing chunk complete. Statistics:
  - Total processed: 1704
  - Overall CLS Fallbacks: 707 (41.49%)
  - Phrase fallbacks: 707/1704 (41.49%)
  - Passage fallbacks: 0/0 (0.00%)
  - Multi fallbacks: 0/0 (0.00%)
Map:  73%|███████▎  | 1000/1367 [00:01<00:00, 555.69 examples/s]2026-07-04 23:37:42 - qa_preprocessor - INFO - Data preprocessing chunk complete. Statistics:
  - Total processed: 2335
  - Overall CLS Fallbacks: 966 (41.37%)
  - Phrase fallbacks: 966/2335 (41.37%)
  - Passage fallbacks: 0/0 (0.00%)
  - Multi fallbacks: 0/0 (0.00%)
Map:   0%|          | 0/335 [00:00<?, ? examples/s]2026-07-04 23:37:43 - qa_preprocessor - INFO - Data preprocessing chunk complete. Statistics:
  - Total processed: 2893
  - Overall CLS Fallbacks: 1188 (41.06%)
  - Phrase fallbacks: 1188/2893 (41.06%)
  - Passage fallbacks: 0/0 (0.00%)
  - Multi fallbacks: 0/0 (0.00%)
Map: 100%|██████████| 335/3


=== LAUNCHING EXTRACTIVE TRANSFORMER PHRASE QA TRAIN SEQUENCE ===


  0%|          | 0/438 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
 33%|███▎      | 146/438 [02:16<04:26,  1.10it/s]

...

{'eval_loss': '1.451', 'eval_exact_match': '0.6738', 'eval_token_f1': '0.7423', 'eval_answerable_exact_match': '0.5089', 'eval_answerable_token_f1': '0.6227', 'eval_runtime': '9.862', 'eval_samples_per_second': '56.58', 'eval_steps_per_second': '3.549', 'epoch': '3'}



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.l

{'train_runtime': '443', 'train_samples_per_second': '15.81', 'train_steps_per_second': '0.989', 'train_loss': '0.8297', 'epoch': '3'}
Saving optimized phrase QA model weights to results_qa_phrase...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


[CLEANUP] Pruned intermediate checkpoint: results_qa_phrase/checkpoint-438
[SUCCESS] Training execution routine complete.
[SUCCESS] Phrase QA model fine-tuning completed successfully.


In [8]:
import os

def train_passage_qa_model():
    """Fine-tunes the extractive QA model on passage-type spoilers using config.py defaults."""
    print("Starting Task 2 Passage QA model training...")
    cmd = (
        "python main_transformer_task2_train.py "
        "--spoiler_type passage "
        "--train_path data/train.jsonl "
        "--val_path data/validation.jsonl"
    )
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Passage QA model fine-tuning failed.")
    print("[SUCCESS] Passage QA model fine-tuning completed successfully.")

train_passage_qa_model()

Starting Task 2 Passage QA model training...


Loading and filtering passage datasets...
Reading data from data/train.jsonl...
Filtered dataset from 3200 to 1274 samples of type 'passage'.
Reading data from data/validation.jsonl...
Filtered dataset from 800 to 322 samples of type 'passage'.
Tokenizing and mapping text spans using: deepset/roberta-base-squad2


Map:   0%|          | 0/1274 [00:00<?, ? examples/s]2026-07-04 23:46:46 - qa_preprocessor - INFO - Data preprocessing chunk complete. Statistics:
  - Total processed: 2018
  - Overall CLS Fallbacks: 991 (49.11%)
  - Phrase fallbacks: 0/0 (0.00%)
  - Passage fallbacks: 991/2018 (49.11%)
  - Multi fallbacks: 0/0 (0.00%)
Map:  78%|███████▊  | 1000/1274 [00:02<00:00, 409.19 examples/s]2026-07-04 23:46:48 - qa_preprocessor - INFO - Data preprocessing chunk complete. Statistics:
  - Total processed: 2555
  - Overall CLS Fallbacks: 1245 (48.73%)
  - Phrase fallbacks: 0/0 (0.00%)
  - Passage fallbacks: 1245/2555 (48.73%)
  - Multi fallbacks: 0/0 (0.00%)
Map:   0%|          | 0/322 [00:00<?, ? examples/s]2026-07-04 23:46:48 - qa_preprocessor - INFO - Data preprocessing chunk complete. Statistics:
  - Total processed: 3244
  - Overall CLS Fallbacks: 1598 (49.26%)
  - Phrase fallbacks: 0/0 (0.00%)
  - Passage fallbacks: 1598/3244 (49.26%)
  - Multi fallbacks: 0/0 (0.00%)
Map: 100%|██████████| 322


=== LAUNCHING EXTRACTIVE TRANSFORMER PASSAGE QA TRAIN SEQUENCE ===


  0%|          | 0/480 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
  0%|          | 0/44 [00:00<?, ?it/s]

...

{'eval_loss': '2.173', 'eval_exact_match': '0.4978', 'eval_token_f1': '0.6109', 'eval_answerable_exact_match': '0.0744', 'eval_answerable_token_f1': '0.3063', 'eval_runtime': '12.21', 'eval_samples_per_second': '56.42', 'eval_steps_per_second': '3.603', 'epoch': '3'}



Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.l

{'train_runtime': '488.3', 'train_samples_per_second': '15.7', 'train_steps_per_second': '0.983', 'train_loss': '1.529', 'epoch': '3'}
Saving optimized passage QA model weights to results_qa_passage...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


[CLEANUP] Pruned intermediate checkpoint: results_qa_passage/checkpoint-320
[SUCCESS] Training execution routine complete.
[SUCCESS] Passage QA model fine-tuning completed successfully.


In [9]:
import os

def train_multi_seq2seq_model():
    """Fine-tunes the generative T5 Seq2Seq model for resolving multi-sentence spoilers

    using the dataset splits and defaults configured in config.py.
    """
    print("Starting Task 2 Generative Multi Seq2Seq model training (Flan-T5)...")
    
    # We let main_transformer_task2_multi_seq2seq.py automatically read 
    # model_name (google/flan-t5-base) and default lr (5e-5) from the CLI defaults.
    cmd = (
        "python main_transformer_task2_multi_seq2seq.py "
        "--train_path data/train.jsonl "
        "--val_path data/validation.jsonl"
    )
    
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Generative Multi Seq2Seq model fine-tuning failed.")
        
    print("[SUCCESS] Generative Multi Seq2Seq T5 model training completed successfully.")

train_multi_seq2seq_model()

Starting Task 2 Generative Multi Seq2Seq model training (Flan-T5)...


Loading and filtering multi-type datasets...
Reading data from data/train.jsonl...
Filtered dataset from 3200 to 559 samples of type 'multi'.
Reading data from data/validation.jsonl...
Filtered dataset from 800 to 143 samples of type 'multi'.


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1600.71it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



=== LAUNCHING GENERATIVE SEQ2SEQ MULTI-SPOILER TRAIN SEQUENCE ===


  0%|          | 0/210 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
 33%|███▎      | 70/210 [01:08<02:10,  1.07it/s]

...

{'eval_loss': '14.53', 'eval_runtime': '6.46', 'eval_samples_per_second': '22.14', 'eval_steps_per_second': '2.787', 'epoch': '3'}



Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.82s/it]
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].
Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '234.4', 'train_samples_per_second': '7.155', 'train_steps_per_second': '0.896', 'train_loss': '16.61', 'epoch': '3'}
Saving optimized Seq2Seq model weights to results_multi_seq2seq...


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.76s/it]


[CLEANUP] Pruned intermediate checkpoint: results_multi_seq2seq/checkpoint-210
[SUCCESS] Training execution routine complete.
[SUCCESS] Generative Multi Seq2Seq T5 model training completed successfully.


In [16]:
import os

def run_integrated_pipeline():
    """Orchestrates end-to-end inference reading model checkpoints and parameters

    directly from the default PipelineConfig (including the trained T5 weights).
    """
    print("Launching end-to-end Integrated Pipeline Inference with Generative T5...")
    
    # All directories (including results_multi_seq2seq) and strategy (seq2seq)
    # are automatically resolved from your config.py file.
    cmd = (
        "python main_integrated_pipeline.py "
        "--val_path data/validation.jsonl "
        "--t1_checkpoint outputs "
        "--phrase_checkpoint results_qa_phrase "
        "--passage_checkpoint results_qa_passage"
    )
    
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Integrated pipeline inference failed.")
        
    print("[SUCCESS] Final submission 'final_integrated_submission.jsonl' generated.")

run_integrated_pipeline()

Launching end-to-end Integrated Pipeline Inference with Generative T5...


2026-07-05 00:25:26 - integrated_pipeline - INFO - Loading validation data...
2026-07-05 00:25:26 - integrated_pipeline - INFO - Initializing Clickbait Spoiler Integrated Pipeline...
2026-07-05 00:25:26 - integrated_pipeline - INFO - Loading Task 1 classification weights from: outputs


Reading data from data/validation.jsonl...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1626.95it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
2026-07-05 00:25:27 - integrated_pipeline - INFO - Loading Phrase QA weights from: results_qa_phrase
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1615.92it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
2026-07-05 00:25:28 - integrated_pipeline - INFO - Loading Passage QA weights from: results_qa_passage
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1740.91it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
2026-07-05 00:25:29 - integrated_pipeline - INFO - Iterative extraction selected. Binding multi model to passage weights.
Map: 100%|██████████| 800/800 [00:00<00:00, 1658.78 examples/s]
2026-07-05 00:25:43 - integrated_pipeline - INFO - Running Task 1 Model Inference...
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWa


=== TASK 1: CLASSIFICATION PERFORMANCE ===
              precision    recall  f1-score   support

      phrase     0.7492    0.7403    0.7447       335
     passage     0.7311    0.6925    0.7113       322
       multi     0.6524    0.7483    0.6971       143

    accuracy                         0.7225       800
   macro avg     0.7109    0.7270    0.7177       800
weighted avg     0.7247    0.7225    0.7228       800


       TASK 1: CLASSIFICATION PERFORMANCE (Table 1 - Leaderboard Format)        
Overall Accuracy: 0.7225
--------------------------------------------------------------------------------
Spoiler Type         | Precision (Pr.) | Recall (Rec.)   | F1-Score       
--------------------------------------------------------------------------------
Phrase               | 0.7492          | 0.7403          | 0.7447         
Passage              | 0.7311          | 0.6925          | 0.7113         
Multi                | 0.6524          | 0.7483          | 0.6971         

Confu

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1598.55it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-07-05 00:28:10 - integrated_pipeline - INFO - Saving final pipeline output file t

Mean BLEU Score:  0.1934
Mean ROUGE-L Score: 0.4493
Mean METEOR Score:  0.4109
Mean BERTScore:     0.9045

            TASK 2: TEXT GENERATION PERFORMANCE (Table 2 - Grouped)             
Group / Tag     | BLEU-4 (BL4)    | BERTScore (BSc.)   | METEOR (MET)   
--------------------------------------------------------------------------------
All             | 0.1934          | 0.9045             | 0.4109         
Phrase          | 0.1875          | 0.9344             | 0.5055         
Passage         | 0.2389          | 0.8939             | 0.3685         
Multi           | 0.1046          | 0.8581             | 0.2849         
[SUCCESS] Final submission 'final_integrated_submission.jsonl' generated.


In [17]:
import os

def generate_html_report():
    """Generates visual charts and the interactive evaluation HTML report."""
    print("Generating final visual evaluation reports...")
    cmd = (
        "python main_generate_report.py "
        "--predictions_path final_integrated_submission.jsonl"
        " --val_path data/validation.jsonl"
        " --output_dir reports"
    )
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError("Visual report generation failed.")
    print("[SUCCESS] Visualization reports compiled.")

generate_html_report()

Generating final visual evaluation reports...


2026-07-05 00:29:00 - visualization_suite - INFO - Loading files:
 - Ground truth: data/validation.jsonl
 - Predictions: final_integrated_submission.jsonl
2026-07-05 00:29:00 - visualization_suite - INFO - Successfully aligned 800 validation samples.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1623.20it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture;

[SUCCESS] Visualization reports compiled.


In [18]:
import shutil
from pathlib import Path

# Define the source directory to compress and the target archive file name
REPORTS_DIR = Path("reports")
ZIP_OUTPUT_NAME = "clickbait_spoiling_evaluation_reports"

def archive_reports_directory():
    """Compresses the entire reports folder into a single ZIP archive for download."""
    if REPORTS_DIR.exists():
        print(f"Compressing all files inside '{REPORTS_DIR}'...")
        # Generates 'clickbait_spoiling_evaluation_reports.zip' in the active workspace
        archive_path = shutil.make_archive(ZIP_OUTPUT_NAME, "zip", REPORTS_DIR)
        print(f"[SUCCESS] ZIP archive successfully compiled at: {archive_path}")
    else:
        print(f"[ERROR] Directory '{REPORTS_DIR}' not found. Verify report generation ran successfully.")

archive_reports_directory()

Compressing all files inside 'reports'...
[SUCCESS] ZIP archive successfully compiled at: /kaggle/working/clickbait-spoiling-IR-final-project/clickbait_spoiling_evaluation_reports.zip
